# 웹검색을 하는 챗봇 만들기 (Tool Calling)

LLM 은 학습 시점 이후의 정보나 실시간 정보를 모른다. **도구(Tool)** 를 쥐여주면 LLM 이 필요할 때 스스로 그 도구를 호출해 답을 보강할 수 있다.

이 노트북에서 다루는 것:
1. Tavily 검색 API 직접 사용
2. LangChain 에서 Tavily 를 Tool 로 사용
3. LLM 에 도구 바인딩 (`bind_tools`)
4. 도구를 호출하고 결과를 LLM 에 되돌리는 그래프 (chatbot ↔ tools 루프)
5. (부록) `create_react_agent` 로 한 번에 구현

## 환경 변수 준비

이 노트북부터는 실제 LLM 을 호출하므로 API 키가 필요하다.
프로젝트 루트의 `.env` 파일에 키를 넣어두고 `load_dotenv()` 로 불러온다.

```
# .env  (.env.example 을 복사해서 작성)
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...   # 웹검색 노트북에서 필요
```

- OpenAI 키: <https://platform.openai.com/api-keys>
- Tavily 키: <https://app.tavily.com>

In [ ]:
import os
from dotenv import load_dotenv

# 프로젝트 루트의 .env 를 찾아 환경변수로 로드
load_dotenv()

# 필요한 키가 있는지 확인
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"
assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY 가 .env 에 없습니다"
print("환경변수 로드 완료:", "OPENAI_API_KEY", "TAVILY_API_KEY")

## 1. Tavily 검색 API 직접 사용하기

Tavily 는 LLM 친화적인 웹검색 API 다. 세 가지 호출 방식이 있다.
- `search()` : 결과/점수/메타데이터를 담은 dict 반환
- `get_search_context()` : 내용과 출처를 담은 문자열 반환
- `qna_search()` : 질문에 대한 짧은 답변 문자열 반환

In [ ]:
from tavily import TavilyClient

tavily_client = TavilyClient()

response = tavily_client.search("What is AI Agent?", max_results=3)
print(response)

In [ ]:
response["results"]

In [ ]:
context = tavily_client.get_search_context(query="What is AI Agent?")
print(context)

In [ ]:
answer = tavily_client.qna_search(query="What is AI Agent?")
print(answer)

## 2. LangChain 에서 Tavily 를 Tool 로 사용하기

`TavilySearch` 는 Tavily 검색 API 를 LangChain 도구로 감싼 것이다.

주요 파라미터:
- `max_results` (int): 결과 개수
- `topic` (str): "general"(기본) / "news" / "finance"
- `search_depth` (str): "basic"(기본) / "advanced"
- `time_range` (str): "day" / "week" / "month" / "year"
- `include_domains` / `exclude_domains` (list): 포함/제외 도메인

In [ ]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results=3)
tool.invoke("What's a 'node' in LangGraph?")

tool_call 형태로도 호출할 수 있다 (LLM 이 도구를 부를 때 쓰는 형식).

In [ ]:
invoke_with_toolcall = tool.invoke({
    "args": {"query": "What's a 'node' in LangGraph?"},
    "type": "tool_call",
    "id": "foo",
    "name": "tavily_search",
})
print(invoke_with_toolcall.content)

## 3. LLM 에 도구 바인딩하기 (`bind_tools`)

먼저 간단한 계산 도구 두 개로 원리를 본다. `@tool` 데코레이터로 함수를 도구로 만들고, `llm.bind_tools(...)` 로 LLM 에 쥐여주면, LLM 이 질문에 맞는 도구를 **호출 형태(tool_calls)** 로 제안한다.

> 중요: `bind_tools` 는 도구를 *실행*하지 않는다. "이 도구를 이 인자로 부르면 됩니다" 라는 의도만 만든다. 실제 실행은 우리가(또는 ToolNode 가) 해야 한다.

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

tools = [add, multiply]

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools(tools)

query = "What is 3 * 12? Also, what is 11 + 49?"
llm_with_tools.invoke(query).tool_calls

## 4. Tavily 검색을 LLM 에 바인딩하기

In [ ]:
tool = TavilySearch(max_results=2)
tools = [tool]

llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools(tools)

일반 인사는 도구 없이 그냥 답하고, 검색이 필요한 질문엔 도구 호출(tool_calls)을 만든다.

In [ ]:
# 일반 질문 → tool_calls 없음
print("인사:", llm_with_tools.invoke("안녕").tool_calls)
# 검색이 필요한 질문 → tool_calls 생성
print("검색:", llm_with_tools.invoke("What is LangGraph?").tool_calls)

## 5. 웹검색 기반으로 답하는 챗봇 그래프 만들기

구조:
```
START → chatbot ──(도구 필요?)──> tools ──┐
          ▲                                │
          └────────────────────────────────┘
          (도구 결과를 받아 다시 판단) → 도구 불필요하면 END
```

### chatbot 노드
LLM(도구 바인딩본)을 호출한다. 일반 답변이거나 tool_calls 를 만든다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)

### tools 노드

마지막 AIMessage 의 `tool_calls` 를 읽어 실제 도구를 실행하고, 결과를 `ToolMessage` 로 만들어 돌려준다.
원리 이해를 위해 직접 구현하지만, LangGraph 의 `ToolNode(tools)` 로 같은 동작을 한 줄에 쓸 수 있다(주석 참고).

In [ ]:
import json
from langchain_core.messages import ToolMessage
# from langgraph.prebuilt import ToolNode  # 이걸로 아래 클래스 전체를 대체 가능

class BasicToolNode:
    """마지막 AIMessage 에서 요청된 도구를 실행하는 노드"""

    def __init__(self, tools: list) -> None:
        self.tools_by_name = {t.name: t for t in tools}

    def __call__(self, inputs: dict):
        if messages := inputs.get("messages", []):
            message = messages[-1]
        else:
            raise ValueError("No message found in input")

        outputs = []
        for tool_call in message.tool_calls:           # 호출하라고 지정된 도구들
            tool_result = self.tools_by_name[tool_call["name"]].invoke(
                tool_call["args"]
            )
            outputs.append(
                ToolMessage(
                    content=json.dumps(tool_result),
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )
        return {"messages": outputs}

tool_node = BasicToolNode(tools=[tool])
# tool_node = ToolNode(tools=[tool])   # 동일한 동작 (prebuilt)
graph_builder.add_node("tools", tool_node)

### 조건부 라우팅

chatbot 의 마지막 메시지에 tool_calls 가 있으면 `tools` 로, 없으면 `END` 로 보낸다.

In [ ]:
def route_tools(state: State):
    """마지막 메시지에 도구 호출이 있으면 tools 로, 아니면 END 로"""
    if isinstance(state, list):
        ai_message = state[-1]
    elif messages := state.get("messages", []):
        ai_message = messages[-1]
    else:
        raise ValueError(f"No messages found in input state: {state}")

    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        return "tools"
    return END

graph_builder.add_conditional_edges(
    "chatbot",
    route_tools,
    {"tools": "tools", END: END},
)

### 엣지 연결 후 컴파일
`tools → chatbot` 으로 되돌려, 도구 결과를 받아 LLM 이 다음 단계를 다시 판단하게 한다 (루프).

In [ ]:
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

### 실행

In [ ]:
def stream_graph_updates(user_input: str):
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)

stream_graph_updates("What do you know about LangGraph?")

## 부록) `create_react_agent` 로 간단히 구현하기

위에서 직접 만든 chatbot↔tools 루프는 ReAct 에이전트의 전형적인 패턴이다. LangGraph 의 prebuilt `create_react_agent` 를 쓰면 같은 것을 한 줄로 만들 수 있다.

In [ ]:
from langgraph.prebuilt import create_react_agent

tool = TavilySearch(max_results=2)
tools = [tool]

llm = ChatOpenAI(model="gpt-4o")
agent = create_react_agent(llm, tools)

response = agent.invoke({"messages": [{"role": "user", "content": "What is LangGraph?"}]})
print(response["messages"][-1].content)

## 정리

- **Tool** = LLM 이 외부 기능(검색/계산 등)을 호출하는 수단
- `bind_tools` 는 도구 호출 *의도(tool_calls)* 만 만들고, 실제 실행은 tools 노드가 함
- `chatbot ↔ tools` 루프 + 조건부 라우팅 = 기본 ReAct 패턴
- `create_react_agent` = 그 패턴의 prebuilt 단축

다음: State 를 활용해 할 일 목록을 관리하는 챗봇.